In [19]:
import os
import json
from datasets import load_dataset
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import BitsAndBytesConfig
import torch

In [ ]:
MODEL_NAME = "VietAI/gpt-neo-1.3B-vietnamese-news"
DATA_PATH_JSONL = "documents/ViBidLQA_train_local.json"
OUTPUT_DIR = "qlora_lora_adapter"
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 512
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 3e-4
NUM_EPOCHS = 1
WEIGHT_DECAY = 0.01
LR_WARMUP_STEPS = 50
SAVE_TOTAL_LIMIT = 2
SEED = 42

In [21]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [22]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

In [24]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=False,
)

In [25]:
model.resize_token_embeddings(len(tokenizer))

Embedding(60001, 2048)

In [26]:
model = prepare_model_for_kbit_training(model)

In [27]:
lora_rank = 8
lora_alpha = 16
lora_dropout = 0.1
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "fc_out"]

peft_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

In [28]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 1,337,890,816 || trainable%: 0.1763


In [29]:
def prepare_examples_from_vibid(path):
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        first = f.readline().strip()
        f.seek(0)
        try:
            json.loads(first)
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                if "context" in obj and "question" in obj and "abstractive_answer" in obj:
                    prompt = f"Context: {obj['context']}\nQuestion: {obj['question']}\nAnswer:"
                    answer = obj["abstractive_answer"]
                elif "input" in obj and "output" in obj:
                    prompt = obj["input"]
                    answer = obj["output"]
                else:
                    prompt = obj.get("question", obj.get("input", ""))
                    answer = obj.get("abstractive_answer", obj.get("output", ""))
                examples.append({"input": prompt, "output": answer})
        except Exception:
            data = json.load(f)
            for obj in data:
                prompt = obj.get("question", obj.get("input", ""))
                answer = obj.get("abstractive_answer", obj.get("output", ""))
                examples.append({"input": prompt, "output": answer})
    return examples

print("Loading dataset from:", DATA_PATH_JSONL)
examples = prepare_examples_from_vibid(DATA_PATH_JSONL)
print(f"Loaded {len(examples)} examples (showing first 2):")
for ex in examples[:2]:
    print(">>", ex["input"][:200], "=>", ex["output"][:200])

Loading dataset from: documents/ViBidLQA_train_local.json
Loaded 5298 examples (showing first 2):
>> Context: Luật Đấu thầu số 22/2023/QH15 Điều 96. Quy định chuyển tiếp khoản 2 . Dự án đầu tư kinh doanh đã phê duyệt và phát hành hồ sơ mời thầu trước ngày Luật này có hiệu lực thi hành thì tiếp tục tổ => Theo quy định tại Điều 96 khoản 2 Luật Đấu thầu số 22/2023/QH15, Chính phủ là cơ quan có thẩm quyền quy định chi tiết việc áp dụng chuyển tiếp đối với dự án đầu tư kinh doanh.
>> Context: Nghị định hướng dẫn Luật Đấu thầu số 22 Điều 12. Chi phí trong lựa chọn nhà thầu khoản 2 . Chi phí lập, thẩm định các nội dung trong quá trình lựa chọn nhà thầu Trường hợp thuê tư vấn đấu thầ => Theo quy định, việc quản lý và sử dụng chi phí nêu tại các khoản 3, 4, 5, 6 và 7 của Điều này được thực hiện theo hướng dẫn của Bộ Tài chính.


In [30]:
def build_prompt(input_text, output_text=None):
    if output_text is None:
        return input_text
    return f"{input_text}\n\n### Trả lời:\n{output_text}"

In [31]:
def tokenize_fn(example):
    prompt = example["input"]
    answer = example["output"]

    prompt_ids = tokenizer(prompt,
                           truncation=True,
                           max_length=MAX_SOURCE_LENGTH,
                           add_special_tokens=False)["input_ids"]

    answer_ids = tokenizer(answer,
                           truncation=True,
                           max_length=MAX_TARGET_LENGTH,
                           add_special_tokens=False)["input_ids"]

    if tokenizer.eos_token_id is not None:
        answer_ids = answer_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + answer_ids

    total_max = MAX_SOURCE_LENGTH + MAX_TARGET_LENGTH
    if len(input_ids) > total_max:
        excess = len(input_ids) - total_max
        prompt_ids = prompt_ids[excess:]
        input_ids = prompt_ids + answer_ids

    attention_mask = [1] * len(input_ids)

    labels = [-100] * len(prompt_ids) + answer_ids

    pad_len = total_max - len(input_ids)
    if pad_len > 0:
        pad_id = tokenizer.pad_token_id
        if pad_id is None:
            tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
            pad_id = tokenizer.pad_token_id
        input_ids = input_ids + [pad_id] * pad_len
        attention_mask = attention_mask + [0] * pad_len
        labels = labels + [-100] * pad_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [32]:
from datasets import Dataset
ds = Dataset.from_list(examples)
ds_tokenized = ds.map(lambda x: tokenize_fn(x), remove_columns=ds.column_names, batched=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=SAVE_TOTAL_LIMIT,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=LR_WARMUP_STEPS,
)

Map:   0%|          | 0/5298 [00:00<?, ? examples/s]

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)

C:\Users\namkh\AppData\Local\Temp\ipykernel_11560\4266133102.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 60000}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
C:\Users\namkh\AppData\Roaming\Python\Python312\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


In [ ]:
print("Saving LoRA adapters to:", OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)